### RAG Fusion Pipeline - Ensemble Retriever

This notebook demonstrates RAG Fusion using **two retrievers combined via an Ensemble Retriever**.
Instead of generating sub-queries with an LLM, we use two different retrieval strategies on the same vector store and fuse their results using **Reciprocal Rank Fusion (RRF)** built into LangChain's `EnsembleRetriever`.

**Retrievers used:**
- **Similarity Search** - standard cosine similarity retrieval
- **MMR (Maximal Marginal Relevance)** - balances relevance with diversity (`lambda_mult=0.5`)

**Steps covered:**
1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create two retrievers (Similarity + MMR)
5. Apply RAG Fusion via Ensemble Retriever with weights [0.5, 0.5]
6. Augmentation - build context from fused documents
7. Generation - produce a grounded answer using an LLM

### Imports & Setup

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file
load_dotenv()

True

### Step 1 - Load the PDF

`PyPDFLoader` reads the PDF and returns one `Document` object per page.

In [2]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

Large pages are split into smaller, overlapping chunks so that the retriever can surface focused, relevant passages rather than entire pages.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


### Step 3 - Embeddings & Vector Store

Each chunk is converted into a dense vector using OpenAI's `text-embedding-3-small` model and stored in a ChromaDB vector store.
Both retrievers will query this same vector store using different strategies.

In [4]:
embedding_model = OllamaEmbeddings(model="qwen3-embedding:4b")
llm = OllamaLLM(model="deepseek-r1:1.5b", temperature=0)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag_ensemble"
)

print("Vector store created successfully.")

Vector store created successfully.


### Step 4 - Create Two Retrievers

We create two retrievers from the same vector store, each using a different search strategy:
- **Similarity Search** - ranks chunks purely by cosine similarity to the query
- **MMR** - Maximal Marginal Relevance reduces redundancy by penalizing chunks that are too similar to already-selected ones. `lambda_mult=0.5` balances relevance and diversity equally.

In [5]:
# Retriever 1: standard cosine similarity search
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [6]:
# Retriever 2: MMR - promotes diverse results alongside relevant ones
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

### Step 5 - RAG Fusion with Ensemble Retriever

`RAGFusion.from_retrievers` wraps both retrievers inside LangChain's `EnsembleRetriever`.
Equal weights `[0.5, 0.5]` mean both retrievers contribute equally to the final RRF ranking.
When `invoke` is called, the ensemble retriever queries both strategies and merges the results.

In [7]:
rag_fusion = RAGFusion.from_retrievers(
    base_retrievers=[similarity_retriever, mmr_retriever],
    weights=[0.5, 0.5],
    k=3
)

In [8]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# Both retrievers run in parallel; EnsembleRetriever fuses the results via RRF
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need to work deeply with specific documents.
The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize
information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube
transcripts, and other formats, and NotebookLM will treat these as the authoritative knowledge base for
answering questions.

--- Document 2 ---
model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

--- Document 3 ---
How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike gen

### Step 6 - Augmentation

The retrieved chunks are concatenated into a single context string.
This context will be injected into the generation prompt to ground the LLM's answer.

In [9]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need to work deeply with specific documents.
The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize
information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube
transcripts, and other formats, and NotebookLM will treat these as the authoritative knowledge base for
answering questions.

model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike general-purpose AI assistants, NotebookLM grounds all of its responses in the source


### Step 7 - Generation

The context and original query are passed to the LLM via a structured prompt.
The LLM is instructed to answer **only** from the provided context and to say `"I don't know"` if the answer isn't there.

In [11]:
# llm = ChatOpenAI(model="gpt-5-mini")

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response)

NotebookLM retrieves relevant information from uploaded documents by treating them as its authoritative knowledge base. It uses a method called Retrieval-Augmented Generation (RAG), which allows the AI to synthesize information from various sources like PDFs, Google Docs, YouTube transcripts, and others. This process enables it to answer questions based on the content of these documents.
